# Tahap 1: Enam Varian Whisper tanpa Adaptasi
Subbab 3.4, 3.6.4, 3.6.5, dan 3.6.8. Menjawab rumusan masalah pertama dan memilih varian untuk Tahap 2 dan Tahap 3.

```
[1] Transkripsi enam varian    -> data/output/tahap1/mentah/<varian>/<ID>.json, waktu_transkripsi.jsonl
[2] Perhitungan metrik         -> metrik_blok.csv, metrik_rekaman.csv, metrik_varian.csv, rtf_varian.csv
[3] Uji kelayakan pelatihan    -> uji_kelayakan_percobaan.csv, uji_kelayakan.csv
[4] Pemilihan varian           -> pemilihan_varian.json
[5] Analisis                   -> statistik_deskriptif.csv, ambang.csv, wilcoxon.csv,
                                  kesalahan_istilah.csv, kandidat_halusinasi.csv
```

Bagian 1 dan 3 mengukur waktu dan memori, sehingga harus dijalankan di perangkat Tabel 3.2. Sebelum menjalankannya:
- laptop tersambung ke pengisi daya dengan mode daya kinerja terbaik;
- tampilan layar dialihkan ke Radeon 780M, sehingga RTX 3050 hanya dipakai komputasi;
- aplikasi lain ditutup, terutama peramban dan aplikasi yang memakai GPU.

Bagian 1 memakan waktu beberapa jam dan dapat dihentikan lalu dilanjutkan; pasangan ulangan dan rekaman yang sudah selesai tidak diulang.

## 0. Persiapan

In [1]:
# !pip install faster-whisper==1.2.1 jiwer==4.0.0 scipy==1.17.1 peft==0.20.0 accelerate==1.14.0 transformers==4.45.2 soundfile pandas

In [1]:
import torch, ctranslate2, faster_whisper, jiwer, scipy, peft
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("CTranslate2 GPU:", ctranslate2.get_cuda_device_count())
print(faster_whisper.__version__, jiwer.__version__ if hasattr(jiwer, "__version__") else "jiwer ok", scipy.__version__, peft.__version__)

c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.4.1+cu121 True NVIDIA GeForce RTX 3050 6GB Laptop GPU
CTranslate2 GPU: 1
1.2.1 jiwer ok 1.17.1 0.20.0


In [2]:
from pathlib import Path
import sys, os, json, math, datetime, importlib, itertools, subprocess, warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
import pandas as pd
import soundfile as sf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.praproses_teks as pt
import src.metrik as mt
importlib.reload(pt); importlib.reload(mt)

OUT        = ROOT / "data" / "output"
MANIFEST   = OUT / "manifest_rekaman.csv"
GLOSARIUM  = ROOT / "data" / "processed" / "glosarium" / "glosarium_semua.csv"
KAMUS_PATH = ROOT / "data" / "processed" / "kamus_normalisasi.json"
ACUAN_DIR  = OUT / "praproses_teks" / "acuan"
AUDIO_PROC = ROOT / "data" / "processed" / "audio"
SEG_LATIH  = OUT / "segmen_latih.csv"
SEG_AUDIO  = ROOT / "data" / "processed" / "segmen_latih"
T1         = OUT / "tahap1"
MENTAH     = T1 / "mentah"
KERJA      = T1 / "_kerja"
LOG_WAKTU  = T1 / "waktu_transkripsi.jsonl"
for d in (T1, MENTAH, KERJA):
    d.mkdir(parents=True, exist_ok=True)

rel = lambda p: Path(p).relative_to(ROOT).as_posix()
PY = sys.executable

In [3]:
manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False, encoding="utf-8-sig")
uji = manifest[manifest.split == "uji"].sort_values("file_id").reset_index(drop=True)
assert len(uji) == 9, f"himpunan uji berisi {len(uji)} rekaman"

kamus = pt.muat_kamus(KAMUS_PATH)
assert kamus.terkunci, "Kamus normalisasi belum terkunci. Kunci dulu di notebook 08."
glosarium = pt.muat_glosarium(GLOSARIUM)

acuan = {fid: json.loads((ACUAN_DIR / f"{fid}.json").read_text(encoding="utf-8")) for fid in uji.file_id}
assert all(a["split"] == "uji" for a in acuan.values())

audio_ada = {fid: (AUDIO_PROC / f"{fid}.wav").exists() for fid in uji.file_id}
durasi = {}
for fid in uji.file_id:
    if audio_ada[fid]:
        i = sf.info(str(AUDIO_PROC / f"{fid}.wav"))
        durasi[fid] = i.frames / i.samplerate
        assert (i.samplerate, i.channels) == (16000, 1), f"{fid}: {i.samplerate} Hz, {i.channels} kanal"
    else:
        durasi[fid] = float(uji.set_index("file_id").at[fid, "durasi_potong_detik"])
selisih = {fid: abs(durasi[fid] - float(uji.set_index("file_id").at[fid, "durasi_potong_detik"])) for fid in durasi}
assert max(selisih.values()) < 0.5, f"durasi WAV tidak cocok dengan manifest: {selisih}"

ring = pd.DataFrame({
    "mata_kuliah": [acuan[f]["mata_kuliah"] for f in uji.file_id],
    "menit": [round(durasi[f] / 60, 1) for f in uji.file_id],
    "token": [len(acuan[f]["token"]) for f in uji.file_id],
    "G": [len(acuan[f]["G"]) for f in uji.file_id],
    "blok": [math.ceil(durasi[f] / mt.PANJANG_BLOK) for f in uji.file_id],
    "audio": ["ada" if audio_ada[f] else "TIDAK ADA" for f in uji.file_id],
}, index=uji.file_id)
print(ring.to_string())
print(f"\n{len(uji)} rekaman, {sum(durasi.values())/3600:.2f} jam, {ring.token.sum()} token, "
      f"{ring.G.sum()} posisi istilah, {ring.blok.sum()} blok tiga menit")
print(f"Kamus terkunci sejak {kamus.tanggal_kunci}")

          mata_kuliah  menit  token    G  blok audio
file_id                                             
AOK-04            AOK   19.5   1759  286     7   ada
AOK-07            AOK   15.6   1455  168     6   ada
APS-03            APS   10.7   1158  125     4   ada
APS-05            APS    9.7   1039  191     4   ada
IMK-02            IMK   18.7   2311  107     7   ada
JARNIR-03      JARNIR   18.2   1751  108     7   ada
PKK-02            PKK   16.9   1150  110     6   ada
RPL-07            RPL   20.2   2782  160     7   ada
RPL-10            RPL   16.8   2201  151     6   ada

9 rekaman, 2.44 jam, 15606 token, 1406 posisi istilah, 54 blok tiga menit
Kamus terkunci sejak 2026-10-13


In [4]:
def nvsmi(q):
    try:
        out = subprocess.check_output(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"],
                                      stderr=subprocess.DEVNULL)
        return out.decode(errors="replace").strip().splitlines()[0].strip()
    except Exception:
        return None

GPU = {"nama": nvsmi("name"), "vram_total_mb": nvsmi("memory.total"),
       "vram_terpakai_mb": nvsmi("memory.used"), "driver": nvsmi("driver_version")}
print(GPU)
if GPU["vram_terpakai_mb"] and int(GPU["vram_terpakai_mb"]) > 400:
    print("[!] VRAM terpakai sebelum mulai lebih dari 400 MB. Alihkan tampilan ke iGPU dan tutup aplikasi lain.")

{'nama': 'NVIDIA GeForce RTX 3050 6GB Laptop GPU', 'vram_total_mb': '6144', 'vram_terpakai_mb': '0', 'driver': '577.02'}


## 1. Tahap 1: transkripsi enam varian

### 1.1 Konfigurasi decoding (Tabel 3.5)
Parameter di luar Tabel 3.5 dibiarkan pada nilai bawaan faster-whisper 1.2.1 dan dicatat di metadata run. Presisi float16 dipakai untuk seluruh varian.

In [5]:
VARIAN = ["tiny", "base", "small", "medium", "large-v3", "large-v3-turbo"]
KODE = dict(zip(VARIAN, ["T1-A", "T1-B", "T1-C", "T1-D", "T1-E", "T1-F"]))

PARAMETER = {
    "language": "id",
    "task": "transcribe",
    "beam_size": 5,
    "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "vad_filter": True,
    "condition_on_previous_text": False,
    "word_timestamps": False,
}
COMPUTE_TYPE = "float16"
DEVICE = "cuda"

ULANGAN = 3
PEMANASAN_DETIK = 120
FID_PEMANASAN = min(durasi, key=durasi.get)

print(pd.Series({k: str(v) for k, v in PARAMETER.items()} | {"presisi": COMPUTE_TYPE}).to_string())
print(f"\nPemanasan: {PEMANASAN_DETIK} detik pertama {FID_PEMANASAN}; ulangan terukur: {ULANGAN}")

language                                                  id
task                                              transcribe
beam_size                                                  5
temperature                   [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
vad_filter                                              True
condition_on_previous_text                             False
word_timestamps                                        False
presisi                                              float16

Pemanasan: 120 detik pertama APS-05; ulangan terukur: 3


### 1.2 Transkripsi dan pengukuran waktu
Tiap varian dijalankan di proses terpisah (`src/transkripsi.py`). Waktu muat model dan puncak memori GPU dicatat, tetapi tidak masuk RTF. Keluaran mentah ulangan pertama diarsipkan tanpa disunting; ulangan berikutnya hanya dicatat sidik teksnya.

In [7]:
JALANKAN_TRANSKRIPSI = True

if JALANKAN_TRANSKRIPSI:
    assert all(audio_ada.values()), f"WAV tidak ditemukan: {[f for f, a in audio_ada.items() if not a]}"
    jam = sum(durasi.values()) / 3600
    print(f"Audio yang diproses: {jam:.2f} jam x {ULANGAN} ulangan x {len(VARIAN)} varian = {jam*ULANGAN*len(VARIAN):.1f} jam audio\n")
    rekaman = [{"file_id": f, "path": str(AUDIO_PROC / f"{f}.wav"), "durasi": round(durasi[f], 3)}
               for f in uji.file_id]
    for v in VARIAN:
        cfg = {"nama": v, "model": v, "device": DEVICE, "compute_type": COMPUTE_TYPE,
               "parameter": PARAMETER, "ulangan": ULANGAN, "rekaman": rekaman,
               "pemanasan": {"path": str(AUDIO_PROC / f"{FID_PEMANASAN}.wav"), "detik": PEMANASAN_DETIK},
               "arsip_mentah": str(MENTAH / v), "log_waktu": str(LOG_WAKTU)}
        p = KERJA / f"transkripsi_{v}.json"
        p.write_text(json.dumps(cfg, ensure_ascii=False, indent=1), encoding="utf-8")
        print(f"VRAM terpakai sebelum {v}: {nvsmi('memory.used')} MB")
        proses = subprocess.Popen([PY, "-m", "src.transkripsi", str(p)], cwd=ROOT, text=True,
                                  encoding="utf-8", errors="replace", stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
        for baris in proses.stdout:
            print(baris, end="")
        assert proses.wait() == 0, f"worker {v} berhenti dengan galat"
        print()
else:
    print("JALANKAN_TRANSKRIPSI False: memakai hasil yang sudah ada di", rel(T1))

Audio yang diproses: 2.44 jam x 3 ulangan x 6 varian = 43.9 jam audio

VRAM terpakai sebelum tiny: 0 MB
[tiny] 0 sudah ada, 27 dikerjakan
[tiny] model dimuat dalam 2.7 detik
[tiny] pemanasan 120 detik audio selesai dalam 3.5 detik
[tiny] ulangan 1/3 AOK-04       53.7 detik  RTF 0.0458
[tiny] ulangan 1/3 AOK-07       32.2 detik  RTF 0.0345
[tiny] ulangan 1/3 APS-03       11.6 detik  RTF 0.0180
[tiny] ulangan 1/3 APS-05       13.3 detik  RTF 0.0229
[tiny] ulangan 1/3 IMK-02       30.0 detik  RTF 0.0267
[tiny] ulangan 1/3 JARNIR-03    21.3 detik  RTF 0.0196
[tiny] ulangan 1/3 PKK-02       16.0 detik  RTF 0.0158
[tiny] ulangan 1/3 RPL-07       30.3 detik  RTF 0.0249
[tiny] ulangan 1/3 RPL-10       25.1 detik  RTF 0.0249
[tiny] ulangan 2/3 AOK-04       61.9 detik  RTF 0.0528
[tiny] ulangan 2/3 AOK-07       27.4 detik  RTF 0.0294
[tiny] ulangan 2/3 APS-03       11.7 detik  RTF 0.0182
[tiny] ulangan 2/3 APS-05       11.9 detik  RTF 0.0205
[tiny] ulangan 2/3 IMK-02       28.1 detik  RTF 0.0250

In [8]:
waktu = pd.DataFrame([json.loads(b) for b in LOG_WAKTU.read_text(encoding="utf-8").splitlines() if b.strip()])
waktu = waktu[waktu.konfigurasi.isin(VARIAN) & waktu.file_id.isin(uji.file_id)]
lengkap = waktu.groupby("konfigurasi").size().reindex(VARIAN, fill_value=0)
print("Pasangan ulangan x rekaman yang tercatat per varian (harus", ULANGAN * len(uji), "):")
print(lengkap.to_string())
assert (lengkap == ULANGAN * len(uji)).all(), "Bagian 1.2 belum selesai untuk semua varian."
for v in VARIAN:
    hilang = [f for f in uji.file_id if not (MENTAH / v / f"{f}.json").exists()]
    assert not hilang, f"keluaran mentah {v} tidak ada untuk {hilang}"

# Pemeriksaan determinisme: keluaran tiap ulangan seharusnya identik dengan ulangan pertama.
sidik = waktu.pivot_table(index=["konfigurasi", "file_id"], columns="ulangan", values="sidik", aggfunc="first")
beda = sidik[sidik.nunique(axis=1) > 1]
print(f"\nPasangan varian x rekaman yang keluarannya berbeda antarulangan: {len(beda)} dari {len(sidik)}")
if len(beda):
    print(beda.to_string())

Pasangan ulangan x rekaman yang tercatat per varian (harus 27 ):
konfigurasi
tiny              27
base              27
small             27
medium            27
large-v3          27
large-v3-turbo    27

Pasangan varian x rekaman yang keluarannya berbeda antarulangan: 10 dari 54
ulangan                               1                 2                 3
konfigurasi file_id                                                        
base        AOK-04     8420b87f42910d91  aa7dd4ea1800ba4c  1b3492f9b84ff9ba
            JARNIR-03  c20be239436c6118  6235f3b45c249222  820ace62a8c8ff09
            RPL-07     6a577a5066ccbbc3  9daefd88ac048c71  9b2309d29f6bbd83
tiny        AOK-04     1515a6e7609c13d5  b725883d6ff74f66  e5d0fbc7161c18ed
            AOK-07     3623579c7650d22f  f835acde4f4c2c33  b18f7c6006eafb71
            APS-05     fd2ceeb251a3ed39  aaff487aeae8bc4c  2c0cd372009e0983
            IMK-02     0540ed1a300f1e20  d49d4004cae4f2b3  5e12ed3784360cb1
            JARNIR-03  137365f8274f5

## 2. Perhitungan metrik
Prosedur subbab 3.6.8, dengan modul `src/metrik.py` yang dipakai juga oleh Tahap 2 dan Tahap 3.

### 2.1 Normalisasi hipotesis dan penyaringan segmen
Segmen hipotesis yang tidak beririsan waktu dengan satu pun segmen acuan dibuang, sebab bagian rekaman itu tidak memiliki acuan dan setiap katanya akan terhitung sebagai penyisipan. Segmen yang tersisa dinormalisasi satu per satu dengan fungsi yang sama dengan acuan, termasuk perlindungan istilah glosarium mata kuliah bersangkutan.

In [9]:
def muat_mentah(v, fid):
    return json.loads((MENTAH / v / f"{fid}.json").read_text(encoding="utf-8"))

hipotesis, dibuang, baris = {}, {}, []
for v in VARIAN:
    for fid in uji.file_id:
        a = acuan[fid]
        norm = pt.Normalisasi(kamus, glosarium[a["mata_kuliah"]])
        seg = muat_mentah(v, fid)["segmen"]
        pakai, buang = mt.saring_segmen(seg, a["segmen"])
        hipotesis[v, fid] = mt.normalisasi_hipotesis(pakai, norm)
        dibuang[v, fid] = buang
        n_buang = len(mt.normalisasi_hipotesis(buang, norm))
        baris.append({"varian": v, "file_id": fid, "segmen": len(seg), "segmen_dibuang": len(buang),
                      "token_hipotesis": len(hipotesis[v, fid]), "token_dibuang": n_buang})
saring = pd.DataFrame(baris)
saring.to_csv(T1 / "penyaringan_segmen.csv", index=False, encoding="utf-8-sig")
r = saring.groupby("varian")[["segmen", "segmen_dibuang", "token_hipotesis", "token_dibuang"]].sum().reindex(VARIAN)
r["%token_dibuang"] = (100 * r.token_dibuang / (r.token_hipotesis + r.token_dibuang)).round(2)
print(r.to_string())
print("\nToken dibuang per rekaman:")
print(saring.pivot_table(index="file_id", columns="varian", values="token_dibuang", aggfunc="sum")[VARIAN].to_string())

                segmen  segmen_dibuang  token_hipotesis  token_dibuang  %token_dibuang
varian                                                                                
tiny              1501              12            15184             34            0.22
base              1699              12            15766             34            0.22
small             1517              11            15640             29            0.19
medium            1520               9            15649             27            0.17
large-v3          2323              10            15664             31            0.20
large-v3-turbo    2271               9            15697             25            0.16

Token dibuang per rekaman:
varian     tiny  base  small  medium  large-v3  large-v3-turbo
file_id                                                       
AOK-04        0     0      0       0         0               0
AOK-07        0     0      0       0         0               0
APS-03        0     0   

### 2.2 Penyelarasan dan pembagian ke blok tiga menit
Satu penyelarasan tingkat kata per rekaman. Operasinya lalu dibagi ke blok menurut waktu mulai segmen acuan; penyisipan ikut blok kata acuan sesudahnya. WER dan TTER dihitung dari operasi itu, sedangkan CER dihitung dengan jiwer pada teks acuan dan hipotesis tiap blok. Blok dengan G kosong tidak memiliki TTER (subbab 2.8.4).

In [10]:
operasi, baris = {}, []
for v in VARIAN:
    for fid in uji.file_id:
        b, ops = mt.hitung_rekaman(acuan[fid], hipotesis[v, fid], durasi[fid])
        operasi[v, fid] = ops
        for x in b:
            baris.append({"varian": v, "kode": KODE[v], "file_id": fid,
                          "mata_kuliah": acuan[fid]["mata_kuliah"], **x})
blok = pd.DataFrame(baris)
blok.to_csv(T1 / "metrik_blok.csv", index=False, encoding="utf-8-sig")

per_v = blok.groupby("varian").size()
assert (per_v == ring.blok.sum()).all(), per_v
kosong = blok[(blok.varian == VARIAN[0]) & (blok.N == 0)]
tanpa_g = blok[(blok.varian == VARIAN[0]) & (blok.G == 0)]
print(f"{ring.blok.sum()} blok per varian; blok tanpa kata acuan: {len(kosong)}; blok tanpa istilah (TTER tidak terdefinisi): {len(tanpa_g)}")
if len(kosong):
    print(kosong[["file_id", "blok", "mulai", "selesai", "I"]].to_string(index=False))

54 blok per varian; blok tanpa kata acuan: 0; blok tanpa istilah (TTER tidak terdefinisi): 1


### 2.3 Verifikasi penyelarasan terhadap jiwer
WER dari penyelarasan sendiri harus sama dengan WER jiwer pada setiap rekaman. Kesamaan ini bukti bahwa operasi yang dipakai TTER berasal dari penyelarasan yang benar (3.6.8 butir 2).

In [11]:
verif = pd.DataFrame([{"varian": v, "file_id": fid,
                       **mt.periksa_jiwer(acuan[fid]["token"], hipotesis[v, fid], operasi[v, fid])}
                      for v in VARIAN for fid in uji.file_id])
verif.to_csv(T1 / "verifikasi_jiwer.csv", index=False, encoding="utf-8-sig")
print(f"WER sama dengan jiwer: {verif.sama.sum()} dari {len(verif)} pasangan varian x rekaman")
print(f"Selisih mutlak terbesar: {(verif.wer_sendiri - verif.wer_jiwer).abs().max():.2e}")
assert verif.sama.all(), verif[~verif.sama].to_string(index=False)

WER sama dengan jiwer: 54 dari 54 pasangan varian x rekaman
Selisih mutlak terbesar: 0.00e+00


### 2.4 WER, CER, TTER, dan RTF
Penggabungan antarblok dan antarrekaman selalu menjumlahkan pembilang dan penyebut lebih dahulu. Nilai agregat per varian adalah nilai yang dipakai untuk pemilihan varian dan perbandingan ambang; sebaran antarblok dilaporkan di Bagian 5.1.

In [12]:
JUMLAH = ["N", "S", "D", "I", "err_kata", "err_kar", "N_kar", "E_G", "G"]

def gabung(df, by):
    g = df.groupby(by, sort=False)[JUMLAH].sum()
    g["WER"] = g.err_kata / g.N
    g["CER"] = g.err_kar / g.N_kar
    g["TTER"] = g.E_G / g.G
    return g

rekaman_m = gabung(blok, ["varian", "file_id"]).reset_index()
rekaman_m.to_csv(T1 / "metrik_rekaman.csv", index=False, encoding="utf-8-sig")
varian_m = gabung(blok, "varian").reindex(VARIAN)

# RTF: per ulangan dihitung atas seluruh himpunan uji (jumlah waktu proses / jumlah durasi),
# lalu direratakan antarulangan. Minimum dan maksimum diambil antarrekaman dari rerata ulangan.
rtf_u = waktu.groupby(["konfigurasi", "ulangan"]).agg(proses=("proses_s", "sum"), durasi=("durasi_s", "sum"))
rtf_u["rtf"] = rtf_u.proses / rtf_u.durasi
rtf_rek = waktu.groupby(["konfigurasi", "file_id"]).rtf.mean()
sesi = waktu.sort_values("waktu").groupby("konfigurasi").first()
rtf_v = pd.DataFrame({
    "RTF": rtf_u.groupby("konfigurasi").rtf.mean(),
    "RTF_sd": rtf_u.groupby("konfigurasi").rtf.std(ddof=1),
    "RTF_min_rekaman": rtf_rek.groupby("konfigurasi").min(),
    "RTF_maks_rekaman": rtf_rek.groupby("konfigurasi").max(),
    "muat_model_s": sesi.muat_s,
    "vram_puncak_mb": sesi.vram_puncak_mb - sesi.vram_awal_mb,
}).reindex(VARIAN)
rtf_v.to_csv(T1 / "rtf_varian.csv", encoding="utf-8-sig")
rtf_u.reset_index().to_csv(T1 / "rtf_ulangan.csv", index=False, encoding="utf-8-sig")

varian_m = varian_m.join(rtf_v)
varian_m.insert(0, "kode", [KODE[v] for v in VARIAN])
varian_m.to_csv(T1 / "metrik_varian.csv", encoding="utf-8-sig")

tampil = varian_m[["kode", "WER", "CER", "TTER", "RTF", "RTF_sd"]].copy()
for k in ("WER", "CER", "TTER"):
    tampil[k] = (100 * tampil[k]).round(2)
tampil[["RTF", "RTF_sd"]] = tampil[["RTF", "RTF_sd"]].round(4)
print("Agregat per varian (WER, CER, TTER dalam persen):")
print(tampil.to_string())
print("\nTTER per rekaman (persen):")
print((100 * rekaman_m.pivot(index="file_id", columns="varian", values="TTER")[VARIAN]).round(1).to_string())

Agregat per varian (WER, CER, TTER dalam persen):
                kode    WER    CER   TTER     RTF  RTF_sd
varian                                                   
tiny            T1-A  37.40  15.39  57.18  0.0297  0.0042
base            T1-B  22.77   7.76  35.21  0.0246  0.0006
small           T1-C  12.78   4.31  22.12  0.0452  0.0004
medium          T1-D   8.81   3.41  14.58  0.1002  0.0034
large-v3        T1-E   4.63   1.70   5.05  0.1619  0.0098
large-v3-turbo  T1-F   6.98   2.83  10.38  0.0549  0.0004

TTER per rekaman (persen):
varian     tiny  base  small  medium  large-v3  large-v3-turbo
file_id                                                       
AOK-04     93.0  60.1   44.8    27.6       8.4            26.6
AOK-07     71.4  45.8   23.8    20.8       4.2             4.8
APS-03     52.0  28.8   21.6    10.4       0.8             8.8
APS-05     45.5  30.9   20.4    15.7       1.0             7.9
IMK-02     42.1  14.0    5.6     0.9       0.0             0.0
JARNIR-03  55.6  

## 3. Uji kelayakan pelatihan
Subbab 3.6.5. Tiap varian dipasangi modul LoRA sesuai Tabel 3.6, dijalankan beberapa langkah latih atas segmen latih sebenarnya, lalu dicatat puncak memori GPU dan waktu per langkah. Bagian ini tidak bergantung pada Bagian 1 dan 2.

### 3.1 Konfigurasi (Tabel 3.6)
Ukuran batch dicoba dari 4, nilai rujukan Liu, Yang dan Qu (2024), lalu diturunkan. Batch terbesar yang muat menjadi ukuran batch Tahap 3. Gradient checkpointing tidak dipakai, kecuali bila batch 1 pun tidak muat; pemakaiannya dicatat dan harus ditambahkan ke Tabel 3.6 bila varian itu terpilih.

Batch disusun dari segmen latih dengan label terpanjang, karena masukan audio Whisper selalu setara 30 detik sehingga panjang label satu-satunya pembeda kebutuhan memori antarsegmen.

**Kriteria dapat dilatih:** percobaan selesai tanpa kehabisan memori, dan memori yang sudah terpakai sebelum percobaan ditambah puncak memori yang dicadangkan PyTorch tidak melebihi VRAM RTX 3050. Kriteria kedua perlu karena driver Windows dapat meminjam RAM sistem ketika VRAM penuh, sehingga pelatihan tidak berhenti dengan galat tetapi melambat berkali lipat. Untuk mencegahnya sejak awal, atur *NVIDIA Control Panel → Manage 3D Settings → CUDA - Sysmem Fallback Policy* ke *Prefer No Sysmem Fallback* sebelum menjalankan bagian ini.

In [13]:
MODEL_HF = {v: f"openai/whisper-{v}" for v in VARIAN}
LORA = {"r": 32, "alpha": 64, "dropout": 0.0, "target_modules": ["q_proj", "v_proj"]}
LAJU_BELAJAR = 1e-5
EPOCH = 35
BATCH_DICOBA = [4, 2, 1]
LANGKAH_PEMANASAN = 3
LANGKAH_UKUR = 10

seg_latih = pd.read_csv(SEG_LATIH, dtype={"seg_latih_id": str}, encoding="utf-8-sig")
seg_latih = seg_latih[seg_latih.split == "latih"]
N_LATIH = len(seg_latih)
SEG_JSON = KERJA / "segmen_latih.json"
SEG_JSON.write_text(json.dumps(seg_latih[["seg_latih_id", "teks"]].to_dict("records"), ensure_ascii=False),
                    encoding="utf-8")
print(f"LoRA: {LORA}, AdamW lr {LAJU_BELAJAR}")
print(f"Segmen latih: {N_LATIH}; batch dicoba: {BATCH_DICOBA}; langkah: {LANGKAH_PEMANASAN} pemanasan + {LANGKAH_UKUR} diukur")

LoRA: {'r': 32, 'alpha': 64, 'dropout': 0.0, 'target_modules': ['q_proj', 'v_proj']}, AdamW lr 1e-05
Segmen latih: 1001; batch dicoba: [4, 2, 1]; langkah: 3 pemanasan + 10 diukur


### 3.2 Pengukuran per varian
Satu proses terpisah per percobaan (`src/uji_kelayakan.py`). Percobaan yang sudah tercatat tidak diulang; hapus barisnya di `uji_kelayakan_percobaan.csv` bila ingin mengulang.

In [14]:
JALANKAN_UJI_KELAYAKAN = True
PERCOBAAN = T1 / "uji_kelayakan_percobaan.csv"

def muat(r):
    return (r.get("status") == "berhasil"
            and r["vram_terpakai_awal_mb"] + r["puncak_dicadangkan_mb"] <= r["vram_total_mb"])

def coba(v, batch, gc):
    cfg = {"varian": v, "model_id": MODEL_HF[v], "batch": batch, "gradient_checkpointing": gc,
           "lora": LORA, "laju_belajar": LAJU_BELAJAR, "segmen_json": str(SEG_JSON),
           "audio_dir": str(SEG_AUDIO / "latih"),
           "langkah_pemanasan": LANGKAH_PEMANASAN, "langkah_ukur": LANGKAH_UKUR}
    p = KERJA / f"lora_{v}_b{batch}_gc{int(gc)}.json"
    p.write_text(json.dumps(cfg), encoding="utf-8")
    out = subprocess.run([PY, "-m", "src.uji_kelayakan", str(p)], cwd=ROOT, capture_output=True, text=True,
                         encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    tanda = [b for b in out.stdout.splitlines() if b.startswith("###HASIL###")]
    if not tanda:
        return {"varian": v, "batch": batch, "gradient_checkpointing": gc, "status": "gagal",
                "pesan": (out.stderr or out.stdout)[-400:]}
    return json.loads(tanda[-1][len("###HASIL###"):])

lama = pd.read_csv(PERCOBAAN, encoding="utf-8-sig").to_dict("records") if PERCOBAAN.exists() else []
sudah = {(r["varian"], int(r["batch"]), bool(r["gradient_checkpointing"])): r for r in lama}

if JALANKAN_UJI_KELAYAKAN:
    for v in VARIAN:
        rencana = [(b, False) for b in BATCH_DICOBA] + [(1, True)]
        for b, gc in rencana:
            if gc and any(muat(sudah[v, x, False]) for x in BATCH_DICOBA if (v, x, False) in sudah):
                break
            r = sudah.get((v, b, gc))
            if r is None:
                print(f"{v:15s} batch {b} {'dengan' if gc else 'tanpa'} gradient checkpointing ...", end=" ", flush=True)
                r = coba(v, b, gc)
                sudah[v, b, gc] = r
                pd.DataFrame(list(sudah.values())).to_csv(PERCOBAAN, index=False, encoding="utf-8-sig")
                ket = (f"{r['puncak_dicadangkan_mb']} MB, {r['detik_per_langkah']} detik/langkah"
                       if r.get("status") == "berhasil" else r.get("pesan", "")[:120])
                print(f"{r['status']}{'' if muat(r) or r['status'] != 'berhasil' else ' tetapi melebihi VRAM'}: {ket}")
            if muat(r):
                break
else:
    print("JALANKAN_UJI_KELAYAKAN False: memakai", rel(PERCOBAAN))

percobaan = pd.DataFrame(list(sudah.values()))
percobaan["muat_vram"] = [muat(r) for r in percobaan.to_dict("records")]
kolom = ["varian", "batch", "gradient_checkpointing", "status", "muat_vram", "panjang_label_token",
         "vram_terpakai_awal_mb", "puncak_dicadangkan_mb", "vram_total_mb", "detik_per_langkah"]
print(percobaan[[k for k in kolom if k in percobaan]].to_string(index=False, float_format=lambda x: f"{x:g}"))

tiny            batch 4 tanpa gradient checkpointing ... berhasil: 926 MB, 0.124 detik/langkah
base            batch 4 tanpa gradient checkpointing ... berhasil: 1366 MB, 0.246 detik/langkah
small           batch 4 tanpa gradient checkpointing ... berhasil: 3128 MB, 0.713 detik/langkah
medium          batch 4 tanpa gradient checkpointing ... berhasil tetapi melebihi VRAM: 7736 MB, 10.361 detik/langkah
medium          batch 2 tanpa gradient checkpointing ... berhasil: 4722 MB, 1.178 detik/langkah
large-v3        batch 4 tanpa gradient checkpointing ... memori habis: OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 byt
large-v3        batch 2 tanpa gradient checkpointing ... berhasil tetapi melebihi VRAM: 8502 MB, 11.587 detik/langkah
large-v3        batch 1 tanpa gradient checkpointing ... berhasil tetapi melebihi VRAM: 6026 MB, 1.704 detik/langkah
large-v3        batch 1 dengan gradient checkpointing ... berhasil: 3854

### 3.3 Ringkasan
Estimasi lama Tahap 3 = ⌈segmen latih / batch⌉ × 35 epoch × detik per langkah, belum termasuk evaluasi validasi tiap 200 langkah.

In [15]:
baris = []
for v in VARIAN:
    p = percobaan[(percobaan.varian == v) & percobaan.muat_vram]
    if len(p):
        r = p.sort_values(["gradient_checkpointing", "batch"], ascending=[True, False]).iloc[0]
        langkah = math.ceil(N_LATIH / r.batch) * EPOCH
        baris.append({"varian": v, "dapat_dilatih": True, "batch": int(r.batch),
                      "gradient_checkpointing": bool(r.gradient_checkpointing),
                      "puncak_memori_mb": int(r.puncak_dicadangkan_mb),
                      "detik_per_langkah": r.detik_per_langkah,
                      "parameter_terlatih": int(r.parameter_terlatih),
                      "%_terlatih": round(100 * r.parameter_terlatih / r.parameter_total, 3),
                      "ukuran_adapter_mb": r.ukuran_adapter_mb,
                      "langkah_tahap3": langkah, "estimasi_jam_tahap3": round(langkah * r.detik_per_langkah / 3600, 1)})
    else:
        baris.append({"varian": v, "dapat_dilatih": False})
kelayakan = pd.DataFrame(baris).set_index("varian")
kelayakan.to_csv(T1 / "uji_kelayakan.csv", encoding="utf-8-sig")
print(kelayakan.to_string())
if kelayakan.get("gradient_checkpointing", pd.Series(dtype=bool)).fillna(False).any():
    print("\n[!] Ada varian yang hanya muat dengan gradient checkpointing. Bila terpilih, tambahkan ke Tabel 3.6.")

                dapat_dilatih  batch  gradient_checkpointing  puncak_memori_mb  detik_per_langkah  parameter_terlatih  %_terlatih  ukuran_adapter_mb  langkah_tahap3  estimasi_jam_tahap3
varian                                                                                                                                                                                   
tiny                     True      4                   False               926              0.124              589824       1.538               2.26            8785                  0.3
base                     True      4                   False              1366              0.246             1179648       1.599               4.52            8785                  0.6
small                    True      4                   False              3128              0.713             3538944       1.443              13.53            8785                  1.7
medium                   True      2                   False          

## 4. Pemilihan varian
Subbab 3.6.5: varian dengan TTER agregat terendah di antara varian yang RTF-nya tidak melebihi satu dan dapat dilatih menurut Bagian 3. Varian yang gugur tetap dilaporkan penuh pada rumusan masalah pertama.

In [16]:
pilih = varian_m[["kode", "TTER", "WER", "RTF"]].join(kelayakan[["dapat_dilatih", "batch"]])
pilih["RTF<=1"] = pilih.RTF <= 1
pilih["lolos"] = pilih["RTF<=1"] & pilih.dapat_dilatih
pilih["alasan_gugur"] = np.select(
    [~pilih["RTF<=1"] & ~pilih.dapat_dilatih, ~pilih["RTF<=1"], ~pilih.dapat_dilatih],
    ["RTF > 1 dan tidak dapat dilatih", "RTF > 1", "tidak dapat dilatih"], "")
assert pilih.lolos.any(), "Tidak ada varian yang memenuhi kedua syarat."
terpilih = pilih[pilih.lolos].TTER.idxmin()
pilih["terpilih"] = pilih.index == terpilih

t = pilih.copy()
t[["TTER", "WER"]] = (100 * t[["TTER", "WER"]]).round(2)
t["RTF"] = t.RTF.round(4)
print(t.to_string())
print(f"\nVarian terpilih untuk Tahap 2 dan Tahap 3: {terpilih} ({KODE[terpilih]}), "
      f"TTER {100*pilih.at[terpilih, 'TTER']:.2f}%, batch Tahap 3 = {int(kelayakan.at[terpilih, 'batch'])}")

urut = pilih[pilih.lolos].TTER.sort_values()
if len(urut) > 1:
    print(f"Selisih TTER dengan peringkat kedua ({urut.index[1]}): {100*(urut.iloc[1]-urut.iloc[0]):.2f} poin persentase")

(T1 / "pemilihan_varian.json").write_text(json.dumps({
    "terpilih": terpilih, "kode_baseline": KODE[terpilih],
    "kriteria": "TTER agregat terendah di antara varian dengan RTF <= 1 dan dapat dilatih (subbab 3.6.5)",
    "batch_tahap3": int(kelayakan.at[terpilih, "batch"]),
    "gradient_checkpointing": bool(kelayakan.at[terpilih, "gradient_checkpointing"]),
    "tabel": pilih.reset_index().astype({"batch": "Int64"}).to_dict("records"),
}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

                kode   TTER    WER     RTF  dapat_dilatih  batch  RTF<=1  lolos alasan_gugur  terpilih
varian                                                                                                
tiny            T1-A  57.18  37.40  0.0297           True      4    True   True                  False
base            T1-B  35.21  22.77  0.0246           True      4    True   True                  False
small           T1-C  22.12  12.78  0.0452           True      4    True   True                  False
medium          T1-D  14.58   8.81  0.1002           True      2    True   True                  False
large-v3        T1-E   5.05   4.63  0.1619           True      1    True   True                   True
large-v3-turbo  T1-F  10.38   6.98  0.0549           True      1    True   True                  False

Varian terpilih untuk Tahap 2 dan Tahap 3: large-v3 (T1-E), TTER 5.05%, batch Tahap 3 = 1
Selisih TTER dengan peringkat kedua (large-v3-turbo): 5.33 poin persentase


2062

## 5. Analisis Tahap 1
Subbab 3.4, untuk rumusan masalah pertama. Analisis gabungan dengan Tahap 2 dan Tahap 3 dikerjakan pada notebook terakhir.

### 5.1 Statistik deskriptif
Rerata, simpangan baku, minimum, dan maksimum dihitung atas blok tiga menit. Nilai agregat (pembilang dan penyebut dijumlahkan dulu) ditampilkan di sampingnya, dan untuk TTER nilai agregat inilah yang menjadi nilai varian sesuai subbab 2.8.4. Peringkat WER dan TTER ditampilkan berdampingan supaya terlihat bila keduanya mengurutkan varian secara berbeda (Ugan, et al., 2025).

In [17]:
baris = []
for v in VARIAN:
    b = blok[blok.varian == v]
    d = {"varian": v, "kode": KODE[v]}
    for k in ("WER", "CER", "TTER"):
        x = 100 * b[k].dropna()
        d |= {f"{k}_agregat": 100 * varian_m.at[v, k], f"{k}_rerata": x.mean(), f"{k}_sd": x.std(ddof=1),
              f"{k}_min": x.min(), f"{k}_maks": x.max(), f"{k}_n_blok": len(x)}
    d |= {"RTF_rerata": varian_m.at[v, "RTF"], "RTF_sd": varian_m.at[v, "RTF_sd"],
          "RTF_min": varian_m.at[v, "RTF_min_rekaman"], "RTF_maks": varian_m.at[v, "RTF_maks_rekaman"]}
    baris.append(d)
desk = pd.DataFrame(baris).set_index("varian")
desk["peringkat_WER"] = desk.WER_agregat.rank().astype(int)
desk["peringkat_TTER"] = desk.TTER_agregat.rank().astype(int)
desk.round(4).to_csv(T1 / "statistik_deskriptif.csv", encoding="utf-8-sig")

def ringkas(k, desimal=2):
    return desk.apply(lambda r: f"{r[f'{k}_agregat']:.{desimal}f} ({r[f'{k}_rerata']:.{desimal}f} ± "
                                f"{r[f'{k}_sd']:.{desimal}f}; {r[f'{k}_min']:.{desimal}f}-{r[f'{k}_maks']:.{desimal}f})", axis=1)
tabel = pd.DataFrame({"kode": desk.kode, "WER %": ringkas("WER"), "CER %": ringkas("CER"),
                      "TTER %": ringkas("TTER"),
                      "RTF": desk.apply(lambda r: f"{r.RTF_rerata:.4f} ± {r.RTF_sd:.4f}; {r.RTF_min:.4f}-{r.RTF_maks:.4f}", axis=1),
                      "peringkat WER/TTER": desk.peringkat_WER.astype(str) + "/" + desk.peringkat_TTER.astype(str)})
print("Format: agregat (rerata blok ± sd; min-maks). RTF: rerata ulangan ± sd; min-maks antarrekaman.")
with pd.option_context("display.max_colwidth", 60):
    print(tabel.to_string())
print(f"\nJumlah blok: WER/CER {int(desk.WER_n_blok.iloc[0])}, TTER {int(desk.TTER_n_blok.iloc[0])} (blok tanpa istilah dikeluarkan)")
if (desk.peringkat_WER != desk.peringkat_TTER).any():
    print("WER dan TTER mengurutkan varian secara berbeda pada:",
          ", ".join(desk.index[desk.peringkat_WER != desk.peringkat_TTER]))

Format: agregat (rerata blok ± sd; min-maks). RTF: rerata ulangan ± sd; min-maks antarrekaman.
                kode                               WER %                              CER %                               TTER %                             RTF peringkat WER/TTER
varian                                                                                                                                                                             
tiny            T1-A  37.40 (39.08 ± 18.81; 16.31-99.29)  15.39 (17.38 ± 18.90; 3.47-96.66)  57.18 (54.74 ± 25.80; 16.67-100.00)  0.0297 ± 0.0042; 0.0176-0.0535                6/6
base            T1-B   22.77 (24.38 ± 11.90; 9.09-72.14)     7.76 (8.96 ± 7.75; 2.15-46.60)   35.21 (34.93 ± 22.68; 0.00-100.00)  0.0246 ± 0.0006; 0.0187-0.0306                5/5
small           T1-C    12.78 (13.68 ± 7.11; 1.60-30.41)     4.31 (4.94 ± 4.11; 0.45-20.04)    22.12 (19.00 ± 15.57; 0.00-57.89)  0.0452 ± 0.0004; 0.0353-0.0567                4/4
mediu

### 5.2 Perbandingan terhadap ambang
WER terhadap 25 persen (Munteanu, et al., 2006) dan RTF terhadap 1 (Pratap, et al., 2020). TTER tidak memiliki ambang rujukan dan hanya dibandingkan antarvarian.

In [18]:
AMBANG_WER, AMBANG_RTF = 0.25, 1.0
ambang = pd.DataFrame({
    "kode": varian_m.kode,
    "WER %": (100 * varian_m.WER).round(2),
    "WER <= 25%": varian_m.WER <= AMBANG_WER,
    "blok dengan WER <= 25%": blok.groupby("varian").apply(
        lambda b: f"{(b.WER.dropna() <= AMBANG_WER).sum()}/{b.WER.notna().sum()}").reindex(VARIAN),
    "RTF": varian_m.RTF.round(4),
    "RTF <= 1": varian_m.RTF <= AMBANG_RTF,
})
ambang.to_csv(T1 / "ambang.csv", encoding="utf-8-sig")
print(ambang.to_string())

                kode  WER %  WER <= 25% blok dengan WER <= 25%     RTF  RTF <= 1
varian                                                                          
tiny            T1-A  37.40       False                   9/54  0.0297      True
base            T1-B  22.77        True                  33/54  0.0246      True
small           T1-C  12.78        True                  50/54  0.0452      True
medium          T1-D   8.81        True                  52/54  0.1002      True
large-v3        T1-E   4.63        True                  52/54  0.1619      True
large-v3-turbo  T1-F   6.98        True                  53/54  0.0549      True


### 5.3 Uji Wilcoxon dua tingkat
Uji peringkat bertanda Wilcoxon dua sisi untuk setiap pasangan varian pada WER, CER, dan TTER, dengan pasangan berselisih nol dibuang (subbab 2.8.6).

- **Tingkat blok** atas nilai tiap blok tiga menit, dengan ambang p < 0,001.
- **Tingkat konfigurasi** atas nilai agregat tiap rekaman (n = 9). Dengan sembilan pasangan, nilai-p terkecil yang mungkin adalah 2/2⁹ ≈ 0,0039, sehingga ambang 0,001 tidak dapat tercapai. Nilai-p tingkat ini dilaporkan apa adanya sebagai pelengkap.

`W` adalah statistik Persamaan 2.15 (jumlah peringkat bertanda), `selisih_median` adalah median selisih varian A dikurangi varian B dalam poin persentase; nilai negatif berarti varian A lebih rendah kesalahannya.

In [19]:
from scipy.stats import wilcoxon, rankdata
ALFA_BLOK = 0.001

def uji_wilcoxon(x, y):
    d = np.asarray(x, float) - np.asarray(y, float)
    dn = d[d != 0]
    if len(dn) == 0:
        return {"n": len(d), "n_tak_nol": 0, "W": 0.0, "p": 1.0, "selisih_median": 0.0}
    R = rankdata(np.abs(dn))
    p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided").pvalue
    return {"n": len(d), "n_tak_nol": len(dn), "W": float(np.sum(np.sign(dn) * R)), "p": float(p),
            "selisih_median": float(np.median(100 * d))}

baris = []
for tingkat, data, idx in (("blok", blok, ["file_id", "blok"]), ("konfigurasi", rekaman_m, ["file_id"])):
    for k in ("WER", "CER", "TTER"):
        tab = data.pivot_table(index=idx, columns="varian", values=k).dropna()
        for a, b in itertools.combinations(VARIAN, 2):
            baris.append({"tingkat": tingkat, "metrik": k, "varian_A": a, "varian_B": b, **uji_wilcoxon(tab[a], tab[b])})
wil = pd.DataFrame(baris)
wil["signifikan"] = np.where(wil.tingkat == "blok", np.where(wil.p < ALFA_BLOK, "ya", "tidak"), "-")
wil.to_csv(T1 / "wilcoxon.csv", index=False, encoding="utf-8-sig")

for k in ("WER", "CER", "TTER"):
    w = wil[wil.metrik == k]
    t = w.pivot_table(index=["varian_A", "varian_B"], columns="tingkat", values="p", sort=False)
    t["selisih_median"] = w[w.tingkat == "blok"].set_index(["varian_A", "varian_B"]).selisih_median.round(2)
    t["n_blok"] = w[w.tingkat == "blok"].set_index(["varian_A", "varian_B"]).n_tak_nol
    t["signifikan (blok)"] = w[w.tingkat == "blok"].set_index(["varian_A", "varian_B"]).signifikan
    print(f"\n{k}: nilai-p per tingkat")
    print(t[["selisih_median", "n_blok", "blok", "konfigurasi", "signifikan (blok)"]]
          .to_string(formatters={"blok": "{:.2e}".format, "konfigurasi": "{:.4f}".format}))


WER: nilai-p per tingkat
tingkat                  selisih_median  n_blok     blok konfigurasi signifikan (blok)
varian_A varian_B                                                                     
tiny     base                     12.00      53 2.39e-10      0.0039                ya
         small                    20.12      54 1.63e-10      0.0039                ya
         medium                   23.73      54 1.63e-10      0.0039                ya
         large-v3                 25.82      54 1.63e-10      0.0039                ya
         large-v3-turbo           24.50      54 1.63e-10      0.0039                ya
base     small                     9.14      54 1.92e-10      0.0039                ya
         medium                   12.06      54 1.63e-10      0.0039                ya
         large-v3                 14.78      54 1.63e-10      0.0039                ya
         large-v3-turbo           14.06      54 1.63e-10      0.0039                ya
small    medium  

### 5.4 Ekspor kesalahan istilah untuk dilabeli
Subbab 3.4 butir 5. Penggolongan kesalahan istilah dan penentuan halusinasi memerlukan penilaian manusia, sehingga notebook ini hanya menyiapkan tabelnya.

`kesalahan_istilah.csv` memuat satu baris untuk setiap kemunculan istilah glosarium yang terkena sedikitnya satu operasi sunting. Isi kolom `golongan` dengan salah satu: `substitusi_mirip`, `penghapusan`, `transliterasi`, atau `lainnya`. Baris yang seluruh operasinya penghapusan sudah terisi `penghapusan`.

`kandidat_halusinasi.csv` memuat penyisipan beruntun dan segmen hipotesis yang dibuang karena tidak beririsan dengan acuan. Isi kolom `halusinasi` dengan `ya` atau `tidak`.

In [20]:
KONTEKS = 5
MIN_SISIP_BERUNTUN = 5
kanonik = (pd.read_csv(GLOSARIUM, dtype=str, keep_default_na=False, encoding="utf-8-sig")
             .drop_duplicates("entri_id").set_index("entri_id").bentuk_kanonik)

baris_istilah, baris_halu = [], []
for v in VARIAN:
    for fid in uji.file_id:
        a, ops, hyp = acuan[fid], operasi[v, fid], hipotesis[v, fid]
        ref, N = a["token"], len(a["token"])
        tepi = (N - 1) in set(a["G"])
        blok_tok = {k: int(s["start"] // mt.PANJANG_BLOK) for s in a["segmen"] for k in range(s["i0"], s["i1"])}
        for x in a["istilah"]:
            o = mt.rentang_istilah_hipotesis(ops, x["i0"], x["i1"], N, tepi)
            salah = [y for y in o if y.jenis != "C"]
            if not salah:
                continue
            jenis = {j: sum(y.jenis == j for y in salah) for j in "SDI"}
            sekitar = [y for y in ops if x["i0"] - KONTEKS <= y.i_src < x["i1"] + KONTEKS and y.i_res is not None]
            baris_istilah.append({
                "varian": v, "file_id": fid, "mata_kuliah": a["mata_kuliah"], "blok": blok_tok[x["i0"]],
                "i0": x["i0"], "entri_id": x["entri_id"], "bentuk_kanonik": kanonik.get(x["entri_id"], ""),
                "istilah_acuan": " ".join(ref[x["i0"]:x["i1"]]),
                "hipotesis": " ".join(hyp[y.i_res] for y in o if y.i_res is not None),
                "S": jenis["S"], "D": jenis["D"], "I": jenis["I"],
                "konteks_acuan": " ".join(ref[max(0, x["i0"] - KONTEKS):x["i1"] + KONTEKS]),
                "konteks_hipotesis": " ".join(hyp[y.i_res] for y in sekitar),
                "golongan": "penghapusan" if jenis["D"] == len(salah) else "", "catatan": ""})
        # penyisipan beruntun
        run = []
        for y in ops + [mt.Operasi("C", -1, None)]:
            if y.jenis == "I":
                run.append(y)
                continue
            if len(run) >= MIN_SISIP_BERUNTUN:
                k = min(run[0].i_src, N - 1)
                baris_halu.append({"varian": v, "file_id": fid, "sumber": "penyisipan beruntun",
                                   "blok": blok_tok.get(k), "n_kata": len(run),
                                   "teks": " ".join(hyp[z.i_res] for z in run),
                                   "konteks_acuan": " ".join(ref[max(0, k - KONTEKS):k + KONTEKS]),
                                   "halusinasi": "", "catatan": ""})
            run = []
        for s in dibuang[v, fid]:
            baris_halu.append({"varian": v, "file_id": fid, "sumber": "segmen di luar acuan",
                               "blok": int(s["start"] // mt.PANJANG_BLOK), "n_kata": len(s["text"].split()),
                               "teks": s["text"].strip(), "konteks_acuan": f"{s['start']:.2f}-{s['end']:.2f} detik",
                               "halusinasi": "", "catatan": ""})

kes = pd.DataFrame(baris_istilah)
kes.to_csv(T1 / "kesalahan_istilah.csv", index=False, encoding="utf-8-sig")
halu = pd.DataFrame(baris_halu, columns=["varian", "file_id", "sumber", "blok", "n_kata", "teks",
                                         "konteks_acuan", "halusinasi", "catatan"])
halu.to_csv(T1 / "kandidat_halusinasi.csv", index=False, encoding="utf-8-sig")

print("Kemunculan istilah yang salah per varian:")
r = kes.groupby("varian").agg(kemunculan=("entri_id", "size"), terisi_otomatis=("golongan", lambda g: (g != "").sum()),
                              perlu_dilabeli=("golongan", lambda g: (g == "").sum())).reindex(VARIAN, fill_value=0)
r["dari_seluruh_kemunculan"] = sum(len(a["istilah"]) for a in acuan.values())
print(r.to_string())
print("\nKandidat halusinasi per varian:")
print(halu.groupby(["varian", "sumber"]).size().unstack(fill_value=0).reindex(VARIAN, fill_value=0).to_string())
print(f"\nDitulis ke {rel(T1 / 'kesalahan_istilah.csv')} dan {rel(T1 / 'kandidat_halusinasi.csv')}")

Kemunculan istilah yang salah per varian:
                kemunculan  terisi_otomatis  perlu_dilabeli  dari_seluruh_kemunculan
varian                                                                              
tiny                   639              130             509                     1149
base                   409               31             378                     1149
small                  256               18             238                     1149
medium                 180                6             174                     1149
large-v3                60                1              59                     1149
large-v3-turbo         130                3             127                     1149

Kandidat halusinasi per varian:
sumber          penyisipan beruntun  segmen di luar acuan
varian                                                   
tiny                             15                    12
base                              8                    12
small        

## 6. Metadata run

In [21]:
import platform, importlib.metadata as im
def versi(p):
    try:
        return im.version(p)
    except im.PackageNotFoundError:
        return None

meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "Tahap 1: enam varian tanpa adaptasi",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": GPU,
    "varian": VARIAN, "kode": KODE,
    "parameter_decoding": PARAMETER, "compute_type": COMPUTE_TYPE,
    "rtf": {"ulangan": ULANGAN, "pemanasan": {"file_id": FID_PEMANASAN, "detik": PEMANASAN_DETIK}},
    "metrik": {"panjang_blok_detik": mt.PANJANG_BLOK, "ambang_p_blok": ALFA_BLOK},
    "uji_kelayakan": {"lora": LORA, "laju_belajar": LAJU_BELAJAR, "batch_dicoba": BATCH_DICOBA,
                      "langkah_pemanasan": LANGKAH_PEMANASAN, "langkah_ukur": LANGKAH_UKUR,
                      "presisi": "bobot dasar float16 beku, LoRA float32, autocast float16 dengan penskala gradien"},
    "kamus": {"status": kamus.status, "tanggal_kunci": kamus.tanggal_kunci},
    "terpilih": terpilih,
    "versi_pustaka": {p: versi(p) for p in ["faster-whisper", "ctranslate2", "onnxruntime", "jiwer", "scipy",
                                             "numpy", "pandas", "torch", "transformers", "peft", "accelerate",
                                             "tokenizers", "soundfile"]},
}
try:
    import faster_whisper, inspect
    sig = inspect.signature(faster_whisper.WhisperModel.transcribe)
    meta["parameter_bawaan_lain"] = {k: repr(p.default) for k, p in sig.parameters.items()
                                     if k not in PARAMETER and k not in ("self", "audio")
                                     and p.default is not inspect.Parameter.empty}
except ImportError:
    pass
(T1 / "run_meta_tahap1.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

{
  "faster-whisper": "1.2.1",
  "ctranslate2": "4.7.2",
  "onnxruntime": "1.26.0",
  "jiwer": "4.0.0",
  "scipy": "1.17.1",
  "numpy": "2.0.2",
  "pandas": "3.0.3",
  "torch": "2.4.1+cu121",
  "transformers": "4.45.2",
  "peft": "0.20.0",
  "accelerate": "1.14.0",
  "tokenizers": "0.20.0",
  "soundfile": "0.13.1"
}


## Output

| Berkas di `data/output/tahap1/` | Isi | Dipakai di |
|---|---|---|
| `mentah/<varian>/<ID>.json` | Keluaran mentah ulangan pertama, tidak disunting | Pemeriksaan ulang, lampiran |
| `waktu_transkripsi.jsonl`, `rtf_ulangan.csv`, `rtf_varian.csv` | Waktu proses tiap ulangan, RTF, waktu muat, puncak VRAM | Bab 4, analisis sumber daya |
| `metrik_blok.csv`, `metrik_rekaman.csv`, `metrik_varian.csv` | Pembilang, penyebut, WER, CER, TTER per blok, rekaman, varian | Bab 4; baris varian terpilih menjadi baseline Tahap 2 dan 3 |
| `verifikasi_jiwer.csv`, `penyaringan_segmen.csv` | Bukti penyelarasan dan jumlah segmen yang dibuang | Lampiran |
| `uji_kelayakan_percobaan.csv`, `uji_kelayakan.csv` | Hasil uji kelayakan LoRA tiap varian | Bab 4, ukuran batch Tahap 3 |
| `pemilihan_varian.json` | Varian terpilih dan batch Tahap 3 | Tahap 2 dan Tahap 3 |
| `statistik_deskriptif.csv`, `ambang.csv`, `wilcoxon.csv` | Analisis rumusan masalah pertama | Bab 4 |
| `kesalahan_istilah.csv`, `kandidat_halusinasi.csv` | Tabel untuk dilabeli manual | Analisis kesalahan istilah (3.4 butir 5) |